In [ ]:

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", 
               "langchain", "langchain-groq", "langchain-community",
               "langgraph", "chromadb", "sentence-transformers", "pypdf"],
               capture_output=False)

In [ ]:
# Imports

import warnings
warnings.filterwarnings("ignore")

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from sentence_transformers import SentenceTransformer
import chromadb
from pypdf import PdfReader
import datetime

print("All imports successful")

In [ ]:
# LLM

llm = ChatGroq(
    api_key="Your_Groq_key",
    model="llama-3.3-70b-versatile"
)
print("LLM ready")

In [ ]:
# Agents
# Read resume
reader = PdfReader("Neha_Resume.pdf")
full_text = "".join(page.extract_text() for page in reader.pages)
print(f"Resume loaded: {len(full_text)} characters")

# Chunk it
chunks = [full_text[i:i+350] for i in range(0, len(full_text), 350) 
          if full_text[i:i+350].strip()]
print(f"Split into {len(chunks)} chunks")

# Embed
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks).tolist()
print("Embeddings created")

# Store in vector DB
db = chromadb.Client()
collection = db.get_or_create_collection("neha_resume_day3")
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"c{i}" for i in range(len(chunks))]
)
print("RAG ready")

In [ ]:
@tool
def get_date() -> str:
    """Returns today's date. No input needed."""
    return str(datetime.date.today())

@tool
def calculator(expression: str) -> str:
    """Calculates math expression like 1234 * 5678"""
    try:
        return str(eval(expression))
    except:
        return "Invalid expression"

tools = [get_date, calculator]
print("Tools ready:", [t.name for t in tools])

In [ ]:
memory = MemorySaver()
config = {"configurable": {"thread_id": "neha_day3"}}

def get_resume_context(question):
    """Fetch relevant resume chunks for a question"""
    q_embed = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=q_embed, n_results=3)
    return "\n".join(results['documents'][0])

def chat(user_input):
    try:
        # Check if question is about resume
        resume_keywords = ["internship", "project", "skill", "experience", 
                          "work", "built", "tech", "stack", "job", "iit",
                          "gradpipe", "isb", "neha", "resume", "suit", "role"]
        
        is_resume_question = any(
            kw in user_input.lower() for kw in resume_keywords
        )
        
        # If resume question → fetch context and inject into prompt
        if is_resume_question:
            context = get_resume_context(user_input)
            system_content = f"""You are an intelligent assistant for 
Neha Kanwadiya, IIT Bombay Engineering Physics student (2027).

Here is relevant information from her resume:
{context}

Answer the user's question using this resume information.
For date or math questions use your tools.
Be concise and professional."""
        else:
            system_content = """You are an intelligent assistant for 
Neha Kanwadiya, IIT Bombay Engineering Physics student (2027).
Be concise and professional."""

        agent = create_react_agent(
            llm,
            tools,
            checkpointer=memory,
            prompt=SystemMessage(content=system_content)
        )
        
        response = agent.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config
        )
        return response['messages'][-1].content
    
    except Exception as e:
        return f"Error: {str(e)}"

print("Agent ready")

In [ ]:
print("=== RAG Test ===")
print(chat("What internships has Neha done?"))

print("\n=== Memory Test (follow-up) ===")
print(chat("Tell me more about the ISB one"))

print("\n=== Tool Test ===")
print(chat("What is today's date?"))

print("\n=== Reasoning Test ===")
print(chat("Based on her skills what jobs suit her?"))

In [ ]:
print("Neha's AI Assistant — LangChain + RAG + Memory")
print("Type 'quit' to exit\n")

while True:
    user = input("You: ")
    if user.lower() == 'quit':
        print("Exiting.")
        break
    print(f"\nAI: {chat(user)}\n")
    print("─" * 50 + "\n")